# Bronze — NSE Bhavcopy Ingest

Downloads daily NSE Equity Bhavcopy (OHLCV + Delivery %) from NSE archives and writes to `workspace.bronze_swing.nse_bhavcopy`. Idempotent — safe to re-run for any date. Includes retry logic, session management, data validation, audit trail, and quality checks.

In [0]:
# ── IMPORTS + SHARED CONFIG ──────────────────────────────────
# Load shared config by reading the config notebook's .py file and exec'ing it.
# (More reliable than %run on Spark Connect / serverless compute.)
import pathlib

_config_path = pathlib.Path("/Workspace/Users/chandrakanthab@gmail.com/Swing-Institutional-Alpha/shared/config.py")
_exec_ns = globals()
exec(compile(_config_path.read_text(), str(_config_path), "exec"), _exec_ns)

import requests
import pandas as pd
import io
import time
import random
import uuid
import logging
from datetime import datetime, date, timedelta
from typing import Optional, Tuple, Set

from pyspark.sql import functions as F, Row
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    LongType, DateType, TimestampType, BooleanType
)
from delta.tables import DeltaTable

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze.bhavcopy")

print("✅ Imports + shared config loaded")

In [0]:
# ── SCHEMA SETUP ──────────────────────────────────────────────
# Creates Unity Catalog tables if they don't exist. Safe to re-run.

def setup_schemas():
    """Create bronze_swing.nse_bhavcopy and bronze_swing.load_audit."""

    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {SCHEMA_BRONZE}
        COMMENT 'Bronze layer — raw NSE market data'
    """)
    print(f"✅ Schema: {SCHEMA_BRONZE}")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {T_BHAVCOPY} (
            trade_date          DATE        NOT NULL COMMENT 'NSE trading date (partition key)',
            symbol              STRING      NOT NULL COMMENT 'NSE ticker symbol',
            series              STRING      NOT NULL COMMENT 'EQ / BE / BZ',
            open_price          DOUBLE      COMMENT 'Opening price (INR)',
            high_price          DOUBLE      COMMENT 'Day high (INR)',
            low_price           DOUBLE      COMMENT 'Day low (INR)',
            close_price         DOUBLE      NOT NULL COMMENT 'Closing price (INR)',
            last_price          DOUBLE      COMMENT 'Last traded price (INR)',
            prev_close          DOUBLE      COMMENT 'Previous close (INR)',
            total_traded_qty    LONG        COMMENT 'Total shares traded (volume)',
            turnover_lacs       DOUBLE      COMMENT 'Turnover in lakhs (INR)',
            no_of_trades        LONG        COMMENT 'Number of trades executed',
            deliv_qty           LONG        COMMENT 'Shares taken for delivery',
            deliv_per           DOUBLE      COMMENT 'Delivery % = deliv_qty / total * 100',
            price_change_pct    DOUBLE      COMMENT '% change vs prev_close',
            day_range_pct       DOUBLE      COMMENT '(high-low)/close * 100',
            is_bullish          BOOLEAN     COMMENT 'True if close > open',
            _source_file        STRING      COMMENT 'NSE CSV filename downloaded',
            _load_ts            TIMESTAMP   COMMENT 'UTC timestamp of load',
            _batch_id           STRING      COMMENT 'Run ID for lineage tracking'
        )
        USING DELTA
        PARTITIONED BY (trade_date)
        COMMENT 'NSE daily equity bhavcopy with delivery data — Bronze'
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true',
            'delta.enableChangeDataFeed'        = 'true',
            'delta.dataSkippingNumIndexedCols'  = '4',
            'delta.columnMapping.mode'          = 'name'
        )
    """)
    print(f"✅ Table: {T_BHAVCOPY}")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {T_LOAD_AUDIT} (
            trade_date          DATE        NOT NULL COMMENT 'Trading date attempted',
            load_ts             TIMESTAMP   NOT NULL COMMENT 'UTC timestamp of attempt',
            status              STRING      NOT NULL COMMENT 'SUCCESS/FAILED/HOLIDAY/WEEKEND/SKIP',
            source_url          STRING      COMMENT 'URL requested',
            rows_raw            LONG        COMMENT 'Rows in raw CSV before filter',
            rows_series_filter  LONG        COMMENT 'Rows after EQ/BE/BZ filter',
            rows_loaded         LONG        COMMENT 'Rows written to Delta',
            download_secs       DOUBLE      COMMENT 'Download time in seconds',
            error_msg           STRING      COMMENT 'Error detail if FAILED',
            batch_id            STRING      COMMENT 'Batch run UUID'
        )
        USING DELTA
        COMMENT 'Load audit trail — every bhavcopy ingestion attempt'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """)
    print(f"✅ Table: {T_LOAD_AUDIT}")
    print("\n✅ Schema setup complete. Safe to re-run.")

setup_schemas()

In [0]:
# ── NSE BHAVCOPY DOWNLOADER ────────────────────────────────────
# Session management, retry with backoff, primary/fallback URLs

def get_nse_session() -> requests.Session:
    """Create a requests Session primed with NSE cookies."""
    session = requests.Session()
    session.headers.update(NSE_REQUEST_HEADERS)
    try:
        resp = session.get(NSE_HOME_URL, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        logger.info(f"NSE session established. Cookies: {list(session.cookies.keys())}")
        time.sleep(random.uniform(1.5, 3.0))
    except Exception as e:
        logger.warning(f"Could not prime NSE session: {e}")
    return session


def build_urls(trade_date: date) -> list:
    """Return [(label, url)] — primary first, fallback second."""
    ddmmyyyy = trade_date.strftime("%d%m%Y")
    return [
        ("primary",  NSE_BHAVCOPY_PRIMARY.format(ddmmyyyy=ddmmyyyy)),
        ("fallback", NSE_BHAVCOPY_FALLBACK.format(ddmmyyyy=ddmmyyyy)),
    ]


def download_csv(session: requests.Session, url: str) -> Tuple[str, float]:
    """Download one URL. Returns (csv_text, elapsed_seconds)."""
    t0 = time.time()
    resp = session.get(url, timeout=REQUEST_TIMEOUT)
    if resp.status_code == 404:
        raise FileNotFoundError(f"404 Not Found: {url}")
    if resp.status_code == 403:
        raise PermissionError(f"403 Forbidden: {url}")
    resp.raise_for_status()
    content = resp.text
    elapsed = time.time() - t0

    if content.lstrip().startswith("<"):
        raise ValueError(f"Got HTML (not CSV). NSE may be blocking: {url[:80]}")
    if len(content) < 1000:
        raise ValueError(f"Response too small ({len(content)} bytes): {url[:80]}")
    return content, elapsed


def download_bhavcopy(trade_date: date, session: requests.Session) -> Tuple[pd.DataFrame, str, float]:
    """Download + parse NSE bhavcopy for one date. Returns (df, url, elapsed)."""
    url_options = build_urls(trade_date)
    for label, url in url_options:
        logger.info(f"Trying {label} URL for {trade_date}")
        last_exc = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                print(f"   [{label}] attempt {attempt}/{MAX_RETRIES}")
                csv_text, elapsed = download_csv(session, url)
                df = pd.read_csv(io.StringIO(csv_text))
                df.columns = df.columns.str.strip()
                print(f"   ✅ {len(df)} rows in {elapsed:.1f}s")
                return df, url, elapsed
            except FileNotFoundError:
                logger.warning(f"404 on {label} for {trade_date}")
                last_exc = FileNotFoundError(f"404 for {trade_date}")
                break
            except PermissionError as e:
                logger.warning("403 — refreshing session")
                session = get_nse_session()
                last_exc = e
                time.sleep(random.uniform(3, 6))
            except Exception as e:
                last_exc = e
                print(f"   ❌ Error: {e}")
                if attempt < MAX_RETRIES:
                    sleep = RETRY_BASE_DELAY * attempt + random.uniform(1, 2)
                    print(f"   Retrying in {sleep:.1f}s...")
                    time.sleep(sleep)
    if isinstance(last_exc, FileNotFoundError):
        raise FileNotFoundError(f"Bhavcopy not found for {trade_date}. Likely holiday or not yet published.")
    raise Exception(f"All download attempts failed for {trade_date}. Last: {last_exc}")

In [0]:
# ── DATA VALIDATION & TRANSFORMATION ──────────────────────────
# Handles NSE data quirks. Bronze = raw but clean.

RAW_TO_SCHEMA = {
    "SYMBOL":        "symbol",
    "SERIES":        "series",
    "OPEN_PRICE":    "open_price",
    "HIGH_PRICE":    "high_price",
    "LOW_PRICE":     "low_price",
    "CLOSE_PRICE":   "close_price",
    "LAST_PRICE":    "last_price",
    "PREV_CLOSE":    "prev_close",
    "TTL_TRD_QNTY": "total_traded_qty",
    "TURNOVER_LACS": "turnover_lacs",
    "NO_OF_TRADES":  "no_of_trades",
    "DELIV_QTY":     "deliv_qty",
    "DELIV_PER":     "deliv_per",
}

NSE_NULL_VALUES = {"-", " - ", "NA", "N/A", "", "nan", "NaN", "None"}


def safe_numeric(series: pd.Series) -> pd.Series:
    """Convert NSE column to numeric, handling '-', 'NA', etc."""
    cleaned = series.astype(str).str.strip().replace(list(NSE_NULL_VALUES), None)
    return pd.to_numeric(cleaned, errors="coerce")


def validate_and_transform(
    raw_df: pd.DataFrame, trade_date: date, batch_id: str, source_url: str
) -> Tuple[pd.DataFrame, int, int]:
    """Clean, validate, and enrich raw NSE bhavcopy data.

    Returns: (clean_df, rows_raw, rows_after_series_filter)
    """
    df = raw_df.copy()
    rows_raw = len(df)
    print(f"   Transforming {rows_raw} raw rows...")

    # Step 1: Normalize column names
    df.columns = df.columns.str.strip().str.upper()
    ALT_NAMES = {
        "TOTTRDQTY":   "TTL_TRD_QNTY",
        "TOTTRDVAL":   "TURNOVER_LACS",
        "TOTALTRADES": "NO_OF_TRADES",
    }
    df.rename(columns=ALT_NAMES, inplace=True)
    for col in RAW_TO_SCHEMA:
        if col not in df.columns:
            logger.warning(f"Column '{col}' missing — filling with null")
            df[col] = None

    # Step 2: Rename to schema
    df = df.rename(columns=RAW_TO_SCHEMA)
    df = df[[v for v in RAW_TO_SCHEMA.values()]]

    # Step 3: String cleanup
    df["symbol"] = df["symbol"].astype(str).str.strip().str.upper()
    df["series"] = df["series"].astype(str).str.strip().str.upper()

    # Step 4: Series filter
    df = df[df["series"].isin(VALID_SERIES)].copy()
    rows_series = len(df)
    print(f"   After series filter: {rows_series} rows ({rows_raw - rows_series} dropped)")
    if rows_series == 0:
        avail = raw_df.iloc[:, 1].unique().tolist()[:10]
        raise ValueError(f"Zero rows after series filter. Available series: {avail}")

    # Step 5: Numeric conversion
    float_cols = ["open_price", "high_price", "low_price", "close_price",
                  "last_price", "prev_close", "turnover_lacs", "deliv_per"]
    int_cols = ["total_traded_qty", "no_of_trades", "deliv_qty"]
    for col in float_cols:
        df[col] = safe_numeric(df[col])
    for col in int_cols:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.strip().replace(list(NSE_NULL_VALUES), None),
            errors="coerce"
        ).astype("Int64")

    # Step 6: Price sanity checks
    bad_price = (
        df["close_price"].isna() |
        (df["close_price"] <= 0) |
        df["high_price"].isna() |
        df["low_price"].isna() |
        (df["high_price"] < df["low_price"])
    )
    n_bad = bad_price.sum()
    if n_bad > 0:
        logger.warning(f"Dropping {n_bad} rows with invalid price data")
        df = df[~bad_price].copy()

    # Step 7: Delivery % validation
    missing_per = (
        df["deliv_per"].isna() &
        df["deliv_qty"].notna() &
        df["total_traded_qty"].notna() &
        (df["total_traded_qty"] > 0)
    )
    df.loc[missing_per, "deliv_per"] = (
        df.loc[missing_per, "deliv_qty"].astype(float) /
        df.loc[missing_per, "total_traded_qty"].astype(float) * 100
    ).round(2)
    df["deliv_per"] = df["deliv_per"].clip(lower=0, upper=100)

    # Step 8: Deduplication
    n_before = len(df)
    df = df.drop_duplicates(subset=["symbol", "series"], keep="last")
    if len(df) < n_before:
        logger.warning(f"Removed {n_before - len(df)} duplicate symbol+series rows")

    # Step 9: Derived columns
    df["trade_date"] = trade_date
    df["price_change_pct"] = ((df["close_price"] - df["prev_close"]) / df["prev_close"] * 100).round(4)
    df["day_range_pct"] = ((df["high_price"] - df["low_price"]) / df["close_price"] * 100).round(4)
    df["is_bullish"] = df["close_price"] > df["open_price"]

    # Step 10: Audit columns
    df["_source_file"] = source_url.rsplit("/", 1)[-1]
    df["_load_ts"] = datetime.utcnow()
    df["_batch_id"] = batch_id

    FINAL_COLS = [
        "trade_date", "symbol", "series",
        "open_price", "high_price", "low_price", "close_price",
        "last_price", "prev_close",
        "total_traded_qty", "turnover_lacs", "no_of_trades",
        "deliv_qty", "deliv_per",
        "price_change_pct", "day_range_pct", "is_bullish",
        "_source_file", "_load_ts", "_batch_id",
    ]
    df = df[[c for c in FINAL_COLS if c in df.columns]]
    print(f"   ✅ Transform: {rows_raw} raw → {rows_series} filtered → {len(df)} clean")
    return df, rows_raw, rows_series

In [0]:
# ── DELTA WRITER + AUDIT ──────────────────────────────────────
# Idempotent: DELETE partition → INSERT (replaces, never duplicates)

def write_to_delta(pdf: pd.DataFrame, trade_date: date) -> int:
    """Write pandas DF to Bronze Delta table. Returns rows written."""
    sdf = spark.createDataFrame(pdf)
    sdf = sdf.withColumn("trade_date", F.to_date(F.col("trade_date").cast("string")))
    rows = sdf.count()

    if not spark.catalog.tableExists(T_BHAVCOPY):
        (sdf.write.format("delta").mode("overwrite")
            .partitionBy("trade_date").option("overwriteSchema", "true")
            .saveAsTable(T_BHAVCOPY))
        print(f"   ✅ Delta table created. Written {rows} rows.")
        return rows

    dt = DeltaTable.forName(spark, T_BHAVCOPY)
    existing = spark.sql(
        f"SELECT COUNT(*) AS n FROM {T_BHAVCOPY} WHERE trade_date = '{trade_date}'"
    ).collect()[0]["n"]
    if existing > 0:
        print(f"   🔄 Re-run: deleting {existing} existing rows for {trade_date}")
        dt.delete(F.col("trade_date") == F.lit(str(trade_date)).cast(DateType()))

    (sdf.write.format("delta").mode("append").saveAsTable(T_BHAVCOPY))
    print(f"   ✅ Written {rows} rows for {trade_date}")
    return rows


def write_audit(
    trade_date: date, status: str, batch_id: str,
    source_url: str = "", rows_raw: int = 0, rows_series: int = 0,
    rows_loaded: int = 0, dl_secs: float = 0.0, error_msg: str = "",
):
    """Append one row to the audit table. Always succeeds (best-effort)."""
    try:
        row = Row(
            trade_date=trade_date, load_ts=datetime.utcnow(), status=status,
            source_url=source_url, rows_raw=rows_raw, rows_series_filter=rows_series,
            rows_loaded=rows_loaded, download_secs=float(dl_secs),
            error_msg=error_msg[:1000] if error_msg else "", batch_id=batch_id,
        )
        adf = spark.createDataFrame([row])
        adf = adf.withColumn("trade_date", F.to_date(F.col("trade_date").cast("string")))
        adf.write.format("delta").mode("append").saveAsTable(T_LOAD_AUDIT)
    except Exception as e:
        logger.error(f"Failed to write audit record: {e}")

In [0]:
# ── LOAD ORCHESTRATION ────────────────────────────────────────
# Incremental load, historical backfill, daily scheduled load

def get_loaded_dates() -> Set[date]:
    """Return set of dates already successfully loaded."""
    try:
        rows = spark.sql(f"SELECT DISTINCT trade_date FROM {T_LOAD_AUDIT} WHERE status = 'SUCCESS'").collect()
        return {r["trade_date"] for r in rows}
    except Exception:
        return set()


def ingest_one_date(
    trade_date: date, session: requests.Session, batch_id: str, force: bool = False
) -> str:
    """Full pipeline for one date: Download → Validate → Write → Audit.

    Returns: SUCCESS / SKIP_LOADED / SKIP_WEEKEND / HOLIDAY / FAILED
    """
    ds = trade_date.isoformat()

    if trade_date.weekday() >= 5:
        print(f"   ⏭  {ds} Saturday/Sunday — skip")
        write_audit(trade_date, "SKIP_WEEKEND", batch_id)
        return "SKIP_WEEKEND"

    if not force and trade_date in get_loaded_dates():
        print(f"   ⏭  {ds} already loaded — skip (force=True to reload)")
        return "SKIP_LOADED"

    print(f"\n{'─'*55}")
    print(f"📅 Ingesting {ds}")
    print(f"{'─'*55}")

    rows_raw = rows_series = rows_loaded = 0
    dl_secs = 0.0
    url = ""

    try:
        raw_df, url, dl_secs = download_bhavcopy(trade_date, session)
        rows_raw = len(raw_df)

        clean_df, rows_raw, rows_series = validate_and_transform(raw_df, trade_date, batch_id, url)

        if rows_series < MIN_EXPECTED_EQ_ROWS:
            raise ValueError(
                f"Only {rows_series} rows after filter — expected ≥{MIN_EXPECTED_EQ_ROWS}. "
                f"Possible truncated download or NSE format change."
            )

        rows_loaded = write_to_delta(clean_df, trade_date)
        write_audit(trade_date, "SUCCESS", batch_id, url, rows_raw, rows_series, rows_loaded, dl_secs)
        print(f"✅ {ds} — SUCCESS ({rows_loaded} rows, {dl_secs:.1f}s)")
        return "SUCCESS"

    except FileNotFoundError as e:
        print(f"📆 {ds} — Holiday/unavailable: {e}")
        write_audit(trade_date, "HOLIDAY", batch_id, url, rows_raw, rows_series, 0, dl_secs, str(e))
        return "HOLIDAY"

    except Exception as e:
        print(f"❌ {ds} — FAILED: {e}")
        write_audit(trade_date, "FAILED", batch_id, url, rows_raw, rows_series, rows_loaded, dl_secs, str(e))
        return "FAILED"


def run_historical_backfill(calendar_days: int = HISTORICAL_CALENDAR_DAYS, force: bool = False) -> dict:
    """Download last N calendar days (~60 trading days) of bhavcopy."""
    today = date.today()
    end_date = today - timedelta(days=1)
    start_date = end_date - timedelta(days=calendar_days)
    trading_days = get_trading_days(start_date, end_date)

    batch_id = f"backfill_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
    total = len(trading_days)

    print(f"{'='*55}")
    print(f"🚀 HISTORICAL BACKFILL")
    print(f"   Period      : {start_date} → {end_date}")
    print(f"   Trading days: {total}")
    print(f"   Force reload: {force}")
    print(f"   Batch ID    : {batch_id}")
    print(f"{'='*55}")

    session = get_nse_session()
    counts = {"SUCCESS": 0, "SKIP_LOADED": 0, "SKIP_WEEKEND": 0, "HOLIDAY": 0, "FAILED": 0}
    failed_dates = []

    for i, td in enumerate(trading_days, 1):
        print(f"\n[{i:>3}/{total}]", end=" ")
        status = ingest_one_date(td, session, batch_id, force=force)
        counts[status] = counts.get(status, 0) + 1
        if status == "FAILED":
            failed_dates.append(td)
        if status not in ("SKIP_LOADED", "SKIP_WEEKEND"):
            time.sleep(random.uniform(*INTER_REQUEST_DELAY))
        if i % SESSION_REFRESH_EVERY == 0 and status not in ("SKIP_LOADED", "SKIP_WEEKEND"):
            print("\n🔄 Refreshing NSE session...")
            session = get_nse_session()

    print(f"\n{'='*55}")
    print(f"📊 BACKFILL COMPLETE")
    print(f"{'='*55}")
    for k, v in counts.items():
        icon = {"SUCCESS": "✅", "SKIP_LOADED": "⏭ ", "HOLIDAY": "📆", "FAILED": "❌", "SKIP_WEEKEND": "⏭ "}.get(k, "  ")
        print(f"   {icon} {k:<15} : {v}")
    if failed_dates:
        print(f"\n⚠️  FAILED DATES — retry with force=True:")
        for d in failed_dates:
            print(f"      {d.isoformat()}")
    return counts


def get_effective_trading_date() -> date:
    """Find the most recent COMPLETED trading day whose data is available.

    Rules:
      - Trading day + after 4:00 PM IST → today (market closed at 3:30, data published by 4 PM)
      - Trading day + before 4:00 PM IST → last trading day (market in progress)
      - Weekend / holiday → last trading day
    """
    import pytz
    ist = pytz.timezone("Asia/Kolkata")
    now_ist = datetime.now(ist)
    today = now_ist.date()

    market_data_available = now_ist.hour >= 16  # 4:00 PM IST

    if is_trading_day(today) and market_data_available:
        return today

    # Walk backwards to find the most recent trading day
    effective = today - timedelta(days=1)
    while not is_trading_day(effective):
        effective -= timedelta(days=1)
    return effective


def run_daily_load():
    """Daily entry point. Downloads the most recent COMPLETED trading day's data.

    Smart date selection:
      - Mon 11 AM → Friday's data (market still open)
      - Mon 4:01 PM → Monday's data (market closed, data available)
      - Sat/Sun → Friday's data
      - Holiday → last trading day's data
    Always reloads (force=True) — deletes existing data and rewrites. No duplicates.
    """
    import pytz
    ist = pytz.timezone("Asia/Kolkata")
    now_ist = datetime.now(ist)
    today = now_ist.date()
    effective_date = get_effective_trading_date()

    print(f"🔄 Daily bhavcopy load — {today} ({now_ist.strftime('%H:%M IST')})")

    if effective_date == today:
        print(f"   Today is a trading day, market closed → downloading today's data")
    else:
        if not is_trading_day(today):
            reason = "weekend" if today.weekday() >= 5 else f"holiday ({NSE_HOLIDAYS.get(today, '?')})"
            print(f"   Today is {reason} → downloading last trading day's data")
        else:
            print(f"   Today is a trading day but market still open → downloading last completed trading day")

    print(f"   📅 Effective date: {effective_date}")

    batch_id = f"daily_{today.strftime('%Y%m%d')}_{now_ist.strftime('%H%M%S')}"
    session = get_nse_session()
    status = ingest_one_date(effective_date, session, batch_id, force=True)

    if status == "SUCCESS":
        print(f"\n🔧 Optimizing partition {effective_date}...")
        spark.sql(f"OPTIMIZE {T_BHAVCOPY} WHERE trade_date = '{effective_date}'")
        run_quality_check(effective_date)
        print_daily_summary(effective_date)

In [0]:
# ── DATA QUALITY CHECKS ───────────────────────────────────────
# Run after every load. Flags issues before Silver layer consumes.

def run_quality_check(trade_date: date = None) -> bool:
    """Validate data integrity for a given date. Returns True if critical checks pass."""
    if trade_date is None:
        trade_date = spark.sql(f"SELECT MAX(trade_date) AS d FROM {T_BHAVCOPY}").collect()[0]["d"]

    print(f"\n🔍 Quality Check — {trade_date}")
    print(f"{'─'*50}")

    stats = spark.sql(f"""
        SELECT
            COUNT(DISTINCT symbol)                                AS total_symbols,
            COUNT(*)                                              AS total_rows,
            SUM(CASE WHEN close_price IS NULL    THEN 1 ELSE 0 END) AS null_close,
            SUM(CASE WHEN close_price <= 0       THEN 1 ELSE 0 END) AS zero_price,
            SUM(CASE WHEN high_price < low_price THEN 1 ELSE 0 END) AS inverted_hl,
            SUM(CASE WHEN deliv_per > 100        THEN 1 ELSE 0 END) AS del_over_100,
            SUM(CASE WHEN deliv_per IS NULL      THEN 1 ELSE 0 END) AS null_deliv,
            ROUND(AVG(deliv_per), 2)                               AS avg_delivery,
            ROUND(MIN(close_price), 2)                             AS min_price,
            ROUND(MAX(close_price), 2)                             AS max_price
        FROM {T_BHAVCOPY}
        WHERE trade_date = '{trade_date}'
        AND   series     = 'EQ'
    """).collect()[0]

    checks = [
        ("Total EQ symbols",       stats["total_symbols"], lambda x: x >= DQ_MIN_EQ_SYMBOLS,     True),
        ("Null close prices",      stats["null_close"],    lambda x: x == 0,                     True),
        ("Zero/neg prices",       stats["zero_price"],    lambda x: x == 0,                     True),
        ("High < Low rows",       stats["inverted_hl"],   lambda x: x == 0,                     True),
        ("Delivery % > 100",      stats["del_over_100"],  lambda x: x == 0,                     True),
        ("Null delivery rows",     stats["null_deliv"],    lambda x: x < DQ_MAX_NULL_DELIVERY,  False),
        ("Avg delivery % (20-80)", stats["avg_delivery"],  lambda x: DQ_AVG_DELIVERY_LO < (x or 0) < DQ_AVG_DELIVERY_HI, False),
    ]

    all_critical_pass = True
    for name, val, ok_fn, critical in checks:
        ok = ok_fn(val) if val is not None else False
        icon = "✅" if ok else ("❌" if critical else "⚠️ ")
        label = "(CRITICAL)" if critical and not ok else ""
        print(f"   {icon} {name:<28} : {val}  {label}")
        if critical and not ok:
            all_critical_pass = False

    print(f"\n   Price range   : ₹{stats['min_price']:,.2f} – ₹{stats['max_price']:,.2f}")
    print(f"   Avg delivery  : {stats['avg_delivery']}%")
    print(f"{'─'*50}")
    print("✅ All critical checks passed." if all_critical_pass else "❌ Critical check(s) FAILED.")
    return all_critical_pass


def print_daily_summary(trade_date: date):
    """Quick market summary for the loaded date."""
    print(f"\n📈 Market summary — {trade_date}")
    spark.sql(f"""
        SELECT
            COUNT(DISTINCT symbol)                          AS total_symbols,
            SUM(CASE WHEN is_bullish THEN 1 ELSE 0 END)    AS up_stocks,
            SUM(CASE WHEN NOT is_bullish THEN 1 ELSE 0 END) AS down_stocks,
            ROUND(AVG(deliv_per), 2)                        AS avg_delivery_pct,
            ROUND(AVG(price_change_pct), 3)                 AS avg_price_chg_pct,
            ROUND(SUM(turnover_lacs) / 100, 0)              AS total_turnover_cr
        FROM {T_BHAVCOPY}
        WHERE trade_date = '{trade_date}'
        AND   series     = 'EQ'
    """).show(truncate=False)

In [0]:
# ── MAIN EXECUTION ────────────────────────────────────────────
# Choose the mode for this run. Uncomment exactly ONE block.

# ═══════════════════════════════════════════════════════
# MODE A — FIRST TIME SETUP (run ONCE)
# Creates tables + loads ~60 trading days of history
# Requires NSE website access — run from a job or interactive session
# ═══════════════════════════════════════════════════════
# setup_schemas()
# run_historical_backfill(calendar_days=HISTORICAL_CALENDAR_DAYS, force=False)
# run_quality_check()

# ═══════════════════════════════════════════════════════
# MODE B — DAILY SCHEDULED JOB
# Databricks job calls this every weekday at ~16:45 IST
# ═══════════════════════════════════════════════════════
# run_daily_load()

# ═══════════════════════════════════════════════════════
# MODE C — MANUAL RECOVERY (specific date re-load)
# Use when a date failed or data was corrupted
# ═══════════════════════════════════════════════════════
# recover_date = date(2026, 9, 18)
# session = get_nse_session()
# batch_id = f"recovery_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
# ingest_one_date(recover_date, session, batch_id, force=True)
# run_quality_check(recover_date)

# ═══════════════════════════════════════════════════════
# MODE D — INSPECT BRONZE TABLE
# ═══════════════════════════════════════════════════════
print("=" * 60)
print("📊 EXISTING BRONZE DATA — QUALITY CHECK + SUMMARY")
print("=" * 60)

run_quality_check()

print("\n" + "=" * 60)
print("📈 Daily market summary (last 15 trading days)")
print("=" * 60)
spark.sql(f"""
    SELECT
        trade_date,
        COUNT(DISTINCT symbol) AS symbols,
        ROUND(AVG(deliv_per), 1) AS avg_del_pct,
        SUM(CASE WHEN is_bullish THEN 1 ELSE 0 END) AS up,
        SUM(CASE WHEN NOT is_bullish THEN 1 ELSE 0 END) AS dn,
        ROUND(SUM(turnover_lacs)/100) AS turnover_cr
    FROM {T_BHAVCOPY}
    WHERE series = 'EQ'
    GROUP BY trade_date
    ORDER BY trade_date DESC
    LIMIT 15
""").show()